# Part I. ETL Pipeline for Pre-Processing the Files

## PLEASE RUN THE FOLLOWING CODE FOR PRE-PROCESSING THE FILES

#### Import Python packages 

In [21]:
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv
import pyscopg2

#### Creating list of filepaths to process original event csv data files

In [2]:
print(os.getcwd())

filepath = os.getcwd() + '/event_data'

for root, dirs, files in os.walk(filepath):
    
    file_path_list = glob.glob(os.path.join(root,'*'))


/workspace/home


#### Processing the files to create the data file csv that will be used for Apache Casssandra tables

In [3]:
full_data_rows_list = [] 
    
for f in file_path_list:

    with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
        csvreader = csv.reader(csvfile) 
        next(csvreader)
        
        for line in csvreader:
            full_data_rows_list.append(line) 
            

csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding = 'utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',\
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))


In [4]:
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

6821


# Part II. Complete the Apache Cassandra coding portion of your project. 

## Now you are ready to work with the CSV file titled <font color=red>event_datafile_new.csv</font>, located within the Workspace directory.  The event_datafile_new.csv contains the following columns: 
- artist 
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

The image below is a screenshot of what the denormalized data should appear like in the <font color=red>**event_datafile_new.csv**</font> after the code above is run:<br>

<img src="images/image_event_datafile_new.jpg">

## Begin writing your Apache Cassandra code in the cells below

#### Creating a Cluster

In [5]:
from cassandra.cluster import Cluster
cluster = Cluster()

session = cluster.connect()

#### Create Keyspace

In [6]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS udacity 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)

except Exception as e:
    print(e)

#### Set Keyspace

In [7]:
try:
    session.set_keyspace('udacity')
except Exception as e:
    print(e)

### Now we need to create tables to run the following queries. Remember, with Apache Cassandra you model the database tables on the queries you want to run.

## Create queries to ask the following three questions of the data

### 1. Give me the artist, song title and song's length in the music app history that was heard during  sessionId = 338, and itemInSession  = 4


### 2. Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10, sessionid = 182
    

### 3. Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'




In [8]:
try: 
    session.execute("CREATE TABLE IF NOT EXISTS song_info_by_session_details (sessionId int, \
                                                        ItemInSession int, \
                                                        song text, \
                                                        length float, \
                                                        artist text, \
                                                        PRIMARY KEY (sessionId, ItemInSession)\
                                                        );")
except psycopg2.Error as e: 
    print("Error: Issue creating table")
    print (e)


##### Data modelling for first query:-
For the first query since we have to return the artist, song title and song length I included the respective columns in my table song_info_by_session_details. Now looking at the snapshot of the data it looked like sessionId and ItemInSession columns individually did not have unique values and were unable to identify unique rows by themselves. So, I had to take a combination of sessionId and ItemInSession which helped me in identifying unique rows so I created a composite key consisting of both sessionId and ItemInSession for creating this table. This helps figure out and return the artist, song title and song length for a given sessionId and ItemInSession value later in the query.

In [9]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) 
    for line in csvreader:
        line[5] = float(line[5].strip("()"))
        line[8] = int(line[8].strip("()"))
        line[3] = int(line[3].strip("()"))
        query = "INSERT INTO song_info_by_session_details (sessionId, ItemInSession, song, length, artist)"
        query = query + "VALUES(%s, %s, %s, %s, %s);"
        session.execute(query, (line[8], line[3], line[9], line[5], line[0]))

#### Do a SELECT to verify that the data have been inserted into each table

In [10]:
try:
    rows = session.execute("SELECT * \
                    FROM song_info_by_session_details;")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)

    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

      sessionid  iteminsession  \
0            23              0   
1            23              1   
2            23              2   
3            23              3   
4            23              4   
5            23              5   
6            23              6   
7            23              7   
8            23              8   
9            23              9   
10           23             10   
11           23             11   
12           23             12   
13           23             13   
14           23             14   
15           23             16   
16           23             17   
17           23             18   
18           23             19   
19           23             20   
20           23             21   
21           23             22   
22           23             23   
23           23             24   
24           23             25   
25           23             26   
26           23             27   
27           23             28   
28           2

In [12]:
try:
    rows = session.execute("SELECT artist, song, length \
                    FROM song_info_by_session_details \
                    WHERE sessionId = 338 AND ItemInSession = 4;")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)

    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

      artist                             song      length
0  Faithless  Music Matters (Mark Knight Dub)  495.307312


### COPY AND REPEAT THE ABOVE THREE CELLS FOR EACH OF THE THREE QUESTIONS

In [13]:
try: 
    session.execute("CREATE TABLE IF NOT EXISTS artist_info_by_user_and_session_details (userId int, \
                                                        sessionId int, \
                                                        ItemInSession int, \
                                                        song text, \
                                                        artist text, \
                                                        firstName text, \
                                                        lastName text, \
                                                        PRIMARY KEY ((userId, sessionId), itemInSession)\
                                                        );")
except psycopg2.Error as e: 
    print("Error: Issue creating table")
    print (e)

file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) 
    for line in csvreader:
        line[8] = int(line[8].strip("()"))
        line[3] = int(line[3].strip("()"))
        line[10] = int(line[10].strip("()"))
        query = "INSERT INTO artist_info_by_user_and_session_details (userId, sessionId, itemInSession, song, artist, firstName, lastName)"
        query = query + "VALUES(%s, %s, %s, %s, %s, %s, %s);"
        session.execute(query, (line[10], line[8], line[3], line[9], line[0], line[1], line[4]))

#### Data modelling for second query:- 
For the second query since we have to return the artist, song, first and last name of the user so I included those attributes in the table artist_info_by_user_and_session_details. Now in the question for this query it is mentioned that we need to return the rows for a particular value of userid and sessionId from the query. Looking at the snapshot of the data one can say that the userid and sessionId do not have unique rows but when we combine them we can identify unique rows from the table. However, we also need to order our queries based on itemInSession attribute. This means that the table needs a clustering column itemInSession along with the partition key userid and sessionId by adding PRIMARY KEY ((userId, sessionId), itemInSession) in our create table query. 

#### Do a SELECT to verify that the data have been inserted into each table

In [14]:
try:
    rows = session.execute("SELECT * \
                    FROM artist_info_by_user_and_session_details;")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)

    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

      userid  sessionid  iteminsession  \
0         58        768              0   
1         58        768              1   
2         58        768              2   
3         85        776              2   
4         85        776              3   
5         85        776              4   
6         85        776              5   
7         85        776              6   
8         85        776              8   
9         85        776              9   
10        85        776             10   
11        85        776             11   
12        85        776             12   
13        85        776             13   
14        85        776             14   
15        85        776             15   
16        85        776             16   
17        85        776             17   
18        85        776             18   
19        85        776             19   
20        85        776             20   
21        85        776             21   
22        85        776           

In [15]:
try:
    rows = session.execute("SELECT artist, song, firstName, lastName \
                            FROM artist_info_by_user_and_session_details \
                            WHERE userId = 10 AND sessionId = 182 \
                            ORDER BY itemInSession;")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)

    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

              artist                                               song  \
0   Down To The Bone                                 Keep On Keepin' On   
1       Three Drives                                        Greece 2000   
2  Sebastien Tellier                                          Kilometer   
3      Lonnie Gordon  Catch You Baby (Steve Pitron & Max Sanna Radio...   

  firstname lastname  
0    Sylvie     Cruz  
1    Sylvie     Cruz  
2    Sylvie     Cruz  
3    Sylvie     Cruz  


In [16]:
try: 
    session.execute("CREATE TABLE IF NOT EXISTS user_details_by_song (song text, \
                                                        user_id int, \
                                                        firstName text, \
                                                        lastName text, \
                                                        PRIMARY KEY (song, user_id)\
                                                        );")
except psycopg2.Error as e: 
    print("Error: Issue creating table")
    print (e)

file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) 
    for line in csvreader:
        line[10] = int(line[10].strip("()"))
        query = "INSERT INTO user_details_by_song (song, user_id, firstName, lastName)"
        query = query + "VALUES(%s, %s, %s, %s);"
        session.execute(query, (line[9], line[10], line[1], line[4]))   
        


#### Data modelling for third query:-
For the third query since we need to return the first name and last name for the user for a given song we add the firstName, lastName and song columns in our table user_details_by_song. Since we need to retun these details based on the song that the user listens to the primary key attribute would be song and user_id since many users may have listened to the same song.

#### Do a SELECT to verify that the data have been inserted into each table

In [17]:
try:
    rows = session.execute("SELECT * \
                    FROM user_details_by_song;")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)

    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

                                                   song  user_id   firstname  \
0                                    Wonder What's Next       49       Chloe   
1                                   In The Dragon's Den       49       Chloe   
2                     Too Tough (1994 Digital Remaster)       44      Aleena   
3                   Rio De Janeiro Blue (Album Version)       49       Chloe   
4                                              My Place       15        Lily   
5                                              My Place       73       Jacob   
6                                        The Lucky Ones       24       Layla   
7                                        I Want You Now       80       Tegan   
8                                             Why Worry       88    Mohammad   
9                                 TvÃÂ¡rÃÂ­ v TvÃÂ¡r       97        Kate   
10                     Lord Chancellor's Nightmare Song       85     Kinsley   
11                                      

In [18]:
try:
    rows = session.execute("SELECT firstName, lastName \
                    FROM user_details_by_song \
                    WHERE song = 'All Hands Against His Own';")
    data = [row._asdict() for row in rows]  

    df = pd.DataFrame(data)
    print(df)
except psycopg2.Error as e:
    print("Error: select *")
    print(e)

    firstname lastname
0  Jacqueline    Lynch
1       Tegan   Levine
2        Sara  Johnson


### Drop the tables before closing out the sessions

In [19]:
try:
    session.execute("DROP TABLE IF EXISTS song_info_by_session_details")
except Execption as exep:
    print(exep)
    
try:
    session.execute("DROP TABLE IF EXISTS artist_info_by_user_and_session_details")
except Execption as exep:
    print(exep)
    
try:
    session.execute("DROP TABLE IF EXISTS user_details_by_song")
except Execption as exep:
    print(exep)  

### Close the session and cluster connection¶

In [20]:
session.shutdown()
cluster.shutdown()